In [ ]:
import pandas as pd
from pathlib import Path

path = Path.cwd() / "task_2_data_ex.xlsx"

data = pd.read_excel(path)
display(data.head())



## ACTUAL SOLUTION


In [ ]:
def get_produced_materials(data: pd.DataFrame) -> set:
    """Set of all material IDs that appear as a produced material."""
    return set(data["produced_material"].unique())


def get_fin_materials(data: pd.DataFrame):
    """IDs of all FIN materials."""
    return data.loc[data["produced_material_release_type"] == "FIN", "produced_material"].unique()



def build_adjacency(data: pd.DataFrame, produced_materials: set) -> dict:
    """produced_material -> list of its component children that are themselves produced - not basic materials."""
    edges = data[["produced_material", "component_material"]].drop_duplicates()
    edges = edges[edges["component_material"].isin(produced_materials)]
    return edges.groupby("produced_material")["component_material"].apply(list).to_dict()


def reachable_from(starts, adj: dict) -> set:
    """All produced materials reachable from `starts` (iterative DFS, cycle-safe)."""
    visited, stack = set(), list(starts)
    while stack:
        m = stack.pop()
        if m in visited:
            continue
        visited.add(m)
        stack.extend(adj.get(m, ()))
    return visited


def get_fin_attributes(data: pd.DataFrame, fin_ids) -> pd.DataFrame:
    """
    FIN attributes per plant/month, renamed to the fin_* output columns.
    """
    columns = ["plant_id", "year", "month", "produced_material",
               "produced_material_production_type", "produced_material_release_type",
               "produced_material_quantity"]
    
    return (data.loc[data["produced_material"].isin(fin_ids), columns].drop_duplicates()
                .rename(columns={
                    "produced_material":                 "fin_material_id",
                    "produced_material_release_type":    "fin_material_release_type",
                    "produced_material_production_type": "fin_material_production_type",
                    "produced_material_quantity":        "fin_production_quantity",
                }))


def sum_quantities_yearly(dataframe_with_fin: pd.DataFrame) -> pd.DataFrame:
    """
    Rename the explosion to the output schema and sum the three quantities per
    plant, year, FIN, produced material and component.
    """
    renamed = dataframe_with_fin.rename(columns={
        "plant_id":                          "plant",
        "produced_material":                 "prod_material_id",
        "produced_material_release_type":    "prod_material_release_type",
        "produced_material_production_type": "prod_material_production_type",
        "produced_material_quantity":        "prod_material_production_quantity",
        "component_material":                "component_id",
        "component_material_release_type":   "component_material_release_type",
        "component_material_production_type":"component_material_production_type",
        "component_material_quantity":       "component_consumption_quantity",
    })

    group_cols = [
        "plant", "year",
        "fin_material_id", "fin_material_release_type", "fin_material_production_type",
        "prod_material_id", "prod_material_release_type", "prod_material_production_type",
        "component_id", "component_material_release_type", "component_material_production_type",
    ]

    # dropna=False - ADD/RM components have NaN production_type
    # without this those component rows would be silently dropped.
    summary = renamed.groupby(group_cols, as_index=False, dropna=False).agg(
        fin_production_quantity=("fin_production_quantity", "sum"),
        prod_material_production_quantity=("prod_material_production_quantity", "sum"),
        component_consumption_quantity=("component_consumption_quantity", "sum"),
    )

    column_order = [
        "plant",
        "fin_material_id", "fin_material_release_type", "fin_material_production_type", "fin_production_quantity",
        "prod_material_id", "prod_material_release_type", "prod_material_production_type", "prod_material_production_quantity",
        "component_id", "component_material_release_type", "component_material_production_type", "component_consumption_quantity",
        "year"
    ]
    return summary[column_order]


def build_bom_explosion(data: pd.DataFrame) -> pd.DataFrame:
    produced_materials = get_produced_materials(data)
    fin_ids = get_fin_materials(data)
    adj = build_adjacency(data, produced_materials)

    # PROD sub-assemblies directly under each FIN (the walk starts below the FIN)
    fin_children = (data[data["produced_material"].isin(fin_ids)]
                    .groupby("produced_material")["component_material"]
                    .apply(lambda s: [c for c in s.unique() if c in produced_materials])
                    .to_dict())
    # print("Fin children:")
    # print(fin_children)

    # one (fin, member material) tuple per node in each FIN's tree - all materials with their finals
    pairs = [(int(fin), int(m))
             for fin in fin_ids
             for m in reachable_from(fin_children.get(fin, ()), adj)]
    # print("Pairs:")
    # print(pairs)
    pairs_df = pd.DataFrame(pairs, columns=["_fin_id", "produced_material"])
    # print("Pairs_df:")
    # display(pairs_df)

    # single vectorized gather instead of concats
    exploded = pairs_df.merge(data, on="produced_material", how="inner")

    fin_info = get_fin_attributes(data, fin_ids)
    merged = exploded.merge(
        fin_info,
        left_on=["plant_id", "year", "month", "_fin_id"],
        right_on=["plant_id", "year", "month", "fin_material_id"],
        how="inner",
    )
    return sum_quantities_yearly(merged)


final = build_bom_explosion(data)
display(final)